# High-Performance License Plate Recognition (LPR) System

This pipeline is optimized for **maximum performance**, **accuracy**, and **real-world latency**. It leverages **YOLOv8** for rapid plate localization and **TrOCR** for robust text extraction. It supports semantic interpretation (Yellow, Blue, Red, White) based on plate suffixes.

**Core Stack:**
- Object Detection: YOLOv8
- OCR: TrOCR (HuggingFace)
- CV / Real-time Pipeline: OpenCV
- Inference: Local PyTorch (Auto-detects CUDA / CPU)

In [ ]:
# ==========================================
# 1. IMPORTS & DEPENDENCIES SETUP
# ==========================================
!pip install -q ultralytics transformers kagglehub opencv-python Pillow torch

import os
import cv2
import torch
import re
import json
import time
import numpy as np
import kagglehub
from ultralytics import YOLO
from transformers import TrOCRProcessor, VisionEncoderDecoderModel
from PIL import Image

import warnings
warnings.filterwarnings('ignore')

In [ ]:
# ==========================================
# 2. HARDWARE & DATASET INITIALIZATION
# ==========================================

# Auto-detect GPU for maximum performance
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"[INFO] Hardware selected: {device}")
if device.type == 'cuda':
    print(f"[INFO] GPU Model: {torch.cuda.get_device_name(0)}")

# Download License Plate Dataset
print("[INFO] Downloading dataset...")
dataset_path = kagglehub.dataset_download("fareselmenshawii/license-plate-dataset")
print("[INFO] Dataset files located at:", dataset_path)

In [ ]:
# ==========================================
# 3. MODEL LOADING
# ==========================================
print("[INFO] Loading YOLOv8 architecture...")
# Note: yolov8n provides rapid real-time performance. For a complete system, you'd run:
# yolo_model.train(data=..., epochs=20, imgsz=640)
yolo_model = YOLO('yolov8n.pt') # Lightweight model for high-speed inference

print("[INFO] Loading TrOCR architecture...")
# TrOCR is extremely robust for printed text
ocr_processor = TrOCRProcessor.from_pretrained('microsoft/trocr-small-printed')
ocr_model = VisionEncoderDecoderModel.from_pretrained('microsoft/trocr-small-printed').to(device)

print("[INFO] System models initialized.")

In [ ]:
# ==========================================
# 4. SEMANTICS & POST-PROCESSING ENGINE
# ==========================================

def clean_ocr_text(raw_text):
    """
    Hardens OCR accuracy using structural regex.
    Expected Pattern: [0-9]{2}[A-Z]-[0-9]{4,5}[A-Z]
    Fixes confusion: O<->0, I<->1, B<->8
    """
    text = raw_text.upper()
    text = re.sub(r'\s+', '', text)
    
    # Standard regex enforcing strict sequence formatting
    # Captures: (2 chars) (1 char) (optional hyphen) (4-5 chars) (1 char)
    match = re.search(r'([0-9A-Z]{2})([A-Z0-9])[-]?([0-9A-Z]{4,5})([A-Z0-9])', text)
    if match:
        # Rectify characters based on their positional semantic meaning
        prefix_digits = match.group(1).translate(str.maketrans('OIB', '018'))
        prefix_letter = match.group(2).translate(str.maketrans('018', 'OIB'))
        serial_digits = match.group(3).translate(str.maketrans('OIB', '018'))
        suffix_letter = match.group(4).translate(str.maketrans('018', 'OIB'))
        return f"{prefix_digits}{prefix_letter}-{serial_digits}{suffix_letter}"
    
    # Fallback rigid replacement if string doesn't perfectly match the length/type
    text = text.replace('O', '0').replace('I', '1').replace('B', '8')
    return text

def get_plate_semantic(plate_text):
    """
    Interprets the plate's suffix for metadata typing:
    'V' -> Yellow, 'X' -> Blue, 'D' -> Red, 'T' -> White
    """
    if not plate_text:
        return "", "Unknown"
        
    last_char = plate_text[-1]
    mapping = {
        'V': 'Yellow plate',
        'X': 'Blue plate',
        'D': 'Red plate',
        'T': 'White plate'
    }
    
    if last_char in mapping:
        return last_char, mapping[last_char]
    return last_char, "Unknown plate format"

In [ ]:
# ==========================================
# 5. PIPELINE INFERENCE ENGINE
# ==========================================

def perform_ocr(cropped_plate_bgr):
    """Extract text rapidly using half-precision TrOCR."""
    # BGR to RGB formatting
    rgb_image = cv2.cvtColor(cropped_plate_bgr, cv2.COLOR_BGR2RGB)
    pil_image = Image.fromarray(rgb_image)
    
    # Preprocess & Transfer to GPU
    pixel_values = ocr_processor(pil_image, return_tensors="pt").pixel_values.to(device)
    
    # Infer Text
    with torch.no_grad():
        generated_ids = ocr_model.generate(pixel_values, max_new_tokens=15)
    predicted_text = ocr_processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
    
    return clean_ocr_text(predicted_text)

def extract_plates(image):
    """
    High-level logic bridging YOLO spatial detection with TrOCR extraction.
    """
    # FP16 inference half flag for speed optimization inside YOLO native predict
    results = yolo_model(image, verbose=False, half=(device.type == 'cuda'))
    
    detections = []
    for result in results:
        for box in result.boxes:
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            conf = float(box.conf[0])
            
            if conf < 0.4:
                continue # Thresholding weak localization
                
            cropped_plate = image[y1:y2, x1:x2]
            if cropped_plate.size == 0:
                continue
                
            # Execute Text extraction
            robust_text = perform_ocr(cropped_plate)
            type_code, type_desc = get_plate_semantic(robust_text)
            
            detections.append({
                "box": (x1, y1, x2, y2),
                "plate_text": robust_text,
                "plate_type_code": type_code,
                "plate_type": type_desc,
                "confidence": round(conf, 4)
            })
    return detections

In [ ]:
# ==========================================
# 6. OVERLAYS & EXPORT API
# ==========================================

def draw_annotations(image, detections, fps=None):
    """Generates bounded boxes with semantic color-mapping overlay."""
    canvas = image.copy()
    for det in detections:
        x1, y1, x2, y2 = det['box']
        text = f"{det['plate_text']} [{det['plate_type']}]"
        
        # Color coding mapped strictly to plate classification
        color = (0, 255, 0) # Green (Default)
        if det['plate_type_code'] == 'V': color = (0, 255, 255) # Yellow
        elif det['plate_type_code'] == 'X': color = (255, 0, 0) # Blue
        elif det['plate_type_code'] == 'D': color = (0, 0, 255) # Red
        elif det['plate_type_code'] == 'T': color = (255, 255, 255) # White
        
        cv2.rectangle(canvas, (x1, y1), (x2, y2), color, 3)
        
        # Add text background shadow for high contrast
        cv2.putText(canvas, text, (x1, max(y1-10, 20)), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 0), 4)
        cv2.putText(canvas, text, (x1, max(y1-10, 20)), cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2)
        
    if fps is not None:
        cv2.putText(canvas, f"FPS: {fps:.1f}", (15, 35), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
        
    return canvas

def predict_image(image_path):
    """
    Endpoint 1: Returns highly structured JSON metadata and inferences for an image.
    """
    image = cv2.imread(image_path)
    if image is None:
        return json.dumps({"error": f"Cannot load {image_path}"})
        
    t0 = time.time()
    detections = extract_plates(image)
    latency = time.time() - t0
    
    # Stripping pixel coordinates natively from the JSON response
    payload = []
    for d in detections:
        res = dict(d)
        del res['box']
        payload.append(res)
        
    # Optional rendering
    # annotated_img = draw_annotations(image, detections)
    # cv2.imshow("Inference Engine", annotated_img)
    # cv2.waitKey(0)
    
    return json.dumps({
        "predictions": payload,
        "latency_sec": round(latency, 4)
    }, indent=2)

def process_video(video_path, skip_frames=2):
    """
    Endpoint 2: Real-time video processor. Includes frame dropping for throughput.
    """
    cap = cv2.VideoCapture(video_path)
    frame_id = 0
    
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
            
        if frame_id % skip_frames != 0:
            frame_id += 1
            continue
            
        t0 = time.time()
        detections = extract_plates(frame)
        latency = time.time() - t0
        fps = 1.0 / (latency if latency > 0 else 1e-6)
        
        annotated_frame = draw_annotations(frame, detections, fps=fps)
        cv2.imshow("LPR Video Stream Pipeline", annotated_frame)
        
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break
            
        frame_id += 1
        
    cap.release()
    cv2.destroyAllWindows()
    return json.dumps({"status": "successful"})

def run_webcam(skip_frames=2):
    """
    Endpoint 3: Live inferencing utilizing your hardware's attached camera.
    """
    return process_video(0, skip_frames)


### ⚡ Quick Execution Tests
Uncomment the lines below to start real-time prediction blocks.

In [ ]:
# Example Execution Blocks:

# 1. Test Single Image
# payload = predict_image("path_to_test_image.jpg")
# print(payload)

# 2. Test Video
# process_video("path_to_test_video.mp4", skip_frames=2)

# 3. Run Live Webcam Feed
# run_webcam(skip_frames=3)
